In [ ]:
import joblib

from notebooks.summersoc.globals import PATH_METRICS_DEMO_EXPLORE, PATH_GP_LIST, ITERATE_THROUGH_X_PARTS, \
    SPLIT_DATA_INTO_X_PARTS
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use('default')

from agent.components import RASK
from agent.components.GaussianProcess import GASK
from agent.components.commons import ServiceType, theoretical_param_bounds, ServiceVar, FIG_SIZE_DEFAULT
from agent.components.commons import SloSet

services = [ServiceType.QR, ServiceType.CV, ServiceType.PC]
slos = [SloSet.DEFAULT, SloSet.HIGH_PERF, SloSet.LOW_COST, SloSet.HIGH_QUALITY]

In [ ]:
df_explore = pd.read_csv(PATH_METRICS_DEMO_EXPLORE)
df_explore_preprocessed = RASK.preprocess_data(df_explore)

## **Analyze**: Regression Analysis of Structural Knowledge

### Create Structural Knowledge

Purpose is to identify structural dependencies directly from data

Also helps to determine which metrics to track, and for the tracked ones, which resolution to choose.

Please talk to **Nefeli Marina Rouska** if you find this interesting

In [ ]:
from pgmpy.structure_score import AICGauss
from utils import visualize_DAG

from pgmpy.base import DAG
from pgmpy.causal_discovery import HillClimbSearch

df_filtered = df_explore_preprocessed[df_explore_preprocessed['service_type'] == ServiceType.QR.value]
df_filtered = df_filtered[['cores', 'throughput', 'data_quality']]

scoring_method = AICGauss(data=df_filtered)
estimator = HillClimbSearch(scoring_method=scoring_method)

estimator.fit(X=df_filtered)
dag: DAG = estimator.causal_graph_

regular = '#a1b2ff'  # blue
special = '#c46262'  # red

visualize_DAG(dag, color_map=[regular, special, regular])

### Functional Data Analysis

Note that both variants commonly benefit from removing outliers from the dataset; in our case, it was not needed according to our judgment.



#### Variant 1: Using a Simple Regression Model

Low complexity (TODO: give the asymptotic runtime and the actual physical runtime)

No uncertainty

In [ ]:
from agent.components.RASK import RASK

rm_list = []

for i in range(ITERATE_THROUGH_X_PARTS):
    rm_all_services = {}
    data_ratio = (i + 1) / SPLIT_DATA_INTO_X_PARTS

    draw_figures = i == (ITERATE_THROUGH_X_PARTS / 2) - 1 or i == ITERATE_THROUGH_X_PARTS - 1
    _rm = RASK(show_figures=draw_figures)
    _rm.init_models(df_explore, data_density=data_ratio)

    print(f"Finished fitting {i + 1} / {ITERATE_THROUGH_X_PARTS} of the Regression Models")

# joblib.dump({'gp_list':gp_list}, PATH_GP_LIST)


#### Variant 2: Using a Gaussian Process

Medium complexity

Has uncertainty

In [ ]:
# import plotly.io as pio
#
# print(pio.renderers)
# pio.renderers.default = "chrome"
# import plotly
# import jupyterlab
#
# print(plotly.__version__)
# print(jupyterlab.__version__)

# for type in ['plotly_mimetype', 'jupyterlab', 'nteract', 'vscode',
#          'notebook', 'notebook_connected', 'kaggle', 'azure', 'colab',
#          'cocalc', 'databricks', 'json', 'png', 'jpeg', 'jpg', 'svg',
#          'pdf', 'browser', 'firefox', 'chrome', 'chromium', 'iframe',
#          'iframe_connected', 'sphinx_gallery', 'sphinx_gallery_png']:

In [ ]:
# TODO: Give the option to change between interactive plot and not; the static
#  plot would also always render withing the notebook.

import plotly.io as pio

pio.renderers.default = 'browser'

gp_list = []
lml_history = []

for i in range(ITERATE_THROUGH_X_PARTS):
    gp_all_services = {}
    data_ratio = (i + 1) / SPLIT_DATA_INTO_X_PARTS
    lml_all_service = []

    for s in services:
        # Initialize and train GP model
        draw_figures = 0 #i == (iterate_through_x_parts / 2) - 1 or i == iterate_through_x_parts - 1
        _gp = GASK(s, create_figures=draw_figures, display_figures=draw_figures)
        _gp.init_model(df_explore, data_density=data_ratio)

        _lml = _gp.get_model_lml(s, "max_tp")
        lml_scaled = _lml / data_ratio
        lml_all_service.append(lml_scaled)
        gp_all_services[s] = _gp

    lml_history.append(lml_all_service)
    gp_list.append(gp_all_services)

    print(f"Finished fitting {i + 1} / {ITERATE_THROUGH_X_PARTS} of the Gaussian Processes")

lml_history = np.array(lml_history)
joblib.dump({'gp_list':gp_list}, PATH_GP_LIST)


#### Create LML Figure

In [ ]:
# 1. Define the X-axis (Data Ratios)
data_ratios = np.arange(1, ITERATE_THROUGH_X_PARTS + 1) / SPLIT_DATA_INTO_X_PARTS

# 2. Create the Figure
plt.figure(figsize=FIG_SIZE_DEFAULT)

# Service labels matching your 'services' list order
labels = ['QR Detector', 'CV Analyzer', 'PC Visualizer']
colors = ['#d62728', '#1f77b4', '#2ca02c']  # Red, Blue, Green

for i in range(lml_history.shape[1]):
    plt.plot(data_ratios * (SPLIT_DATA_INTO_X_PARTS / ITERATE_THROUGH_X_PARTS), lml_history[:, i],
             marker='o', markersize=4, linewidth=2,
             label=labels[i], color=colors[i])

# 3. Formatting
abs_samples = int(len(df_explore) * (ITERATE_THROUGH_X_PARTS / SPLIT_DATA_INTO_X_PARTS))
plt.xlabel(f'Ratio of Training Data ({int(abs_samples / 3)} Samples)', fontsize=12)
plt.ylabel('Scaled LML / # Samples', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(title="Service Type")
plt.show()

In [ ]:

RUN_HEATMAP = False  # TODO: Does the heatmap provide some benefit?
if RUN_HEATMAP:

    from agent.components.GaussianProcess import get_empirical_variable_bounds
    from utils import visualize_ndarray
    from agent.components.commons import ServiceVar
    from typing import Dict
    from agent.components.Optimizer import VersatileMapElites


    def extract_pfo_for_SLOs(gp_service, slos: Dict[ServiceVar, float], slo_type: str, simple_bounds):
        v_me = VersatileMapElites(gp_service.s_type, bins=8)

        #  I'm getting the black cells because they are not explored.
        #  What I can do is force all cells to be explored at least once,
        #  or just run gradient descent for each cell multiple (like 5) times.
        v_me.run_search(slos, gp_service, simple_bounds, iterations=2000)
        visualize_ndarray(v_me.fitness_table, gp_service.s_type.value + "_" + slo_type, cmap="viridis")


    final_empirical_bounds = get_empirical_variable_bounds(df_explore_preprocessed)[ServiceType.QR]
    simple_param_bounds = final_empirical_bounds.copy()
    del simple_param_bounds[ServiceVar.PERFORMANCE]
    simple_param_bounds = list(simple_param_bounds.values())

    candidate_solutions = []
    # (1) Here I give it increasingly more training data
    for i in range(ITERATE_THROUGH_X_PARTS):
        candidates = extract_pfo_for_SLOs(gp_list[i][ServiceType.QR], SloSet.DEFAULT.value, "DEFAULT",
                                          simple_param_bounds)
        candidate_solutions.append(candidates)
